# Dialforge Final Release Acceptance v4

This is the **final frozen release gate**, not another tuning benchmark. It runs all three shipping Qwen tiers with their production decoding settings, the full prior adversarial suite plus new brutal objections, deterministic call-control and claim-guard regressions, streaming LLM speed, Faster Whisper STT, Chatterbox Nano TTS, and warmed end-to-end local voice-start latency.

### Before Run All
1. Kaggle Settings → Accelerator → **GPU**. T4 x2 is excellent; the benchmark intentionally uses one visible GPU so the result remains representative of a single-GPU customer PC.
2. Turn **Internet ON**.
3. Click **Run All** and leave the session running. The first run installs a pinned Python 3.11 voice stack and can take a while.

A release PASS requires every shipping model to clear the sales, safety, tool-routing and latency gates. SIP/PSTN carrier latency is intentionally excluded here and remains the final real-phone acceptance check.


In [ ]:
import pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/kaggle/working')
REPO = ROOT / 'Axemetric-Caller-Beta-Runtime'
print('=== DIALFORGE FINAL RELEASE ACCEPTANCE v4 ===')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. Enable a Kaggle GPU accelerator.')
print('GPU(s):\n' + gpu.stdout.strip())
probe = subprocess.run(['curl','-I','-L','--max-time','15','https://github.com'], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Internet appears disabled. Turn Internet ON in Kaggle settings.')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git',str(REPO)], check=True)
sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('Pinned public benchmark source:', sha)


In [ ]:
import subprocess, sys, pathlib
bootstrap = REPO / 'benchmarks' / 'dialforge_final_kaggle_bootstrap_v4.py'
print('Running final gate:', bootstrap)
proc = subprocess.Popen([sys.executable, str(bootstrap)], cwd=str(REPO / 'benchmarks'), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines=[]
for line in proc.stdout:
    print(line, end='')
    lines.append(line)
code = proc.wait()
log = pathlib.Path('/kaggle/working/dialforge-final-release-v4-bootstrap.log')
log.write_text(''.join(lines), encoding='utf-8')
print('\nBenchmark exit code:', code)
print('Bootstrap log:', log)


In [ ]:
import json, pathlib, pandas as pd
report_path = pathlib.Path('/kaggle/working/dialforge-final-release-v4/dialforge-final-release-v4.json')
if not report_path.exists():
    raise RuntimeError('Final report was not produced. Check /kaggle/working/dialforge-final-release-v4-bootstrap.log')
report=json.loads(report_path.read_text(encoding='utf-8'))
rows=[]
for model,d in report['models'].items():
    rows.append({
      'Model':model,
      'PASS':d['passed'],
      'Product Sales':d['sales']['product_score'],
      'Raw Sales':d['sales']['raw_score'],
      'Product Tools':d['tools']['accuracy'],
      'Raw Native Tools':d['tools']['raw_native_accuracy'],
      'Integrity':d['sales']['integrity'],
      'Critical':d['sales']['product_critical'],
      'TTFT median ms':d['stream_speed']['ttft_median_ms'],
      'First sentence p95 ms':d['stream_speed']['first_sentence_p95_ms'],
      'Voice start median ms':d['voice_speed']['voice_start_median_ms'],
      'Voice start p95 ms':d['voice_speed']['voice_start_p95_ms'],
      'STT WER':d['voice_speed']['stt_wer_median'],
      'TTS RTF':d['voice_speed']['tts_rtf_median'],
    })
display(pd.DataFrame(rows))
print('\nGLOBAL REGRESSIONS')
print('Router:', report['regressions']['router']['accuracy'])
print('Claim guard:', report['regressions']['claim_guard']['accuracy'])
print('\nFAILED PRODUCT TOOL CASES')
fails=[]
for model,d in report['models'].items():
    for case in d['tools']['cases']:
        if not case['passed']:
            fails.append({'Model':model,'Case':case['id'],'Source':case['source'],'Tools':', '.join(case['tools']),'Content':case['content']})
display(pd.DataFrame(fails)) if fails else print('None')
print('\nWEAK / CRITICAL SALES TURNS')
weak=[]
for model,d in report['models'].items():
    for conv in d['sales']['conversations']:
        for turn in conv['turns']:
            if float(turn['product_score']) < 85 or int(turn['product_critical']) > 0:
                weak.append({'Model':model,'Conversation':conv['id'],'Expected':turn['expect'],'Product Score':turn['product_score'],'Failures':', '.join(turn['product_failures']),'Answer':turn['product']})
display(pd.DataFrame(weak)) if weak else print('None')
print('\n' + '='*80)
print('FINAL RELEASE GATE:', 'PASS' if report['release_gate_pass'] else 'BLOCKED')
print('='*80)
print('Report:', report_path)
print('Results ZIP: /kaggle/working/dialforge-final-release-v4-results.zip')
